# FINANCE 384 Assignment 1 – Part A

## Task A.4: Hyperparameter Tuning for Gradient Boosting

This standalone notebook tunes the Gradient Boosting Regression model selected in A.3.

The required protocol is:

1. Fit every candidate specification on the **training sample only**.
2. Use the **validation sample only** to select hyperparameters.
3. Rank candidates using pooled validation RMSE, with lower values preferred.
4. Retain the already-fitted winning training model.
5. **Do not re-estimate** the selected model on training + validation.
6. Keep the test period untouched for A.5 and A.6.


### Files required

Upload these files before running:

- `FINANCE384_assignmentA_development_panel.csv`
- `FINANCE384_stock_month_data_dictionary.csv`

`FINANCE384_market.csv` is not required for A.4.


In [1]:
# A.4.1 Imports and reproducibility settings

import itertools
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

PANEL_FILE = "FINANCE384_assignmentA_development_panel.csv"
DICTIONARY_FILE = "FINANCE384_stock_month_data_dictionary.csv"

RANDOM_STATE = 384


In [2]:
# A.4.2 Load the supplied data

panel = pd.read_csv(PANEL_FILE)
data_dictionary = pd.read_csv(DICTIONARY_FILE)

panel["date"] = pd.to_datetime(panel["date"])

print("Panel shape:", panel.shape)
print("Date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("Unique stocks:", panel["permno"].nunique())
print("Duplicate stock-month rows:", panel.duplicated(["permno", "date"]).sum())


Panel shape: (198298, 27)
Date range: 1990-01-31 to 2022-12-30
Unique stocks: 1252
Duplicate stock-month rows: 0


### Feature set

The richer model uses the same base information as the OLS benchmark: 18 numeric stock/market characteristics plus `ff49_code`.

The tuning stage changes only Gradient Boosting hyperparameters; it does not change the underlying predictor information.


In [3]:
# A.4.3 Define predictors

numeric_predictors = [
    "size",
    "bm",
    "mom12_2",
    "vol12",
    "beta60",
    "ivol60",
    "turnover",
    "dollar_volume",
    "amihud_illiq",
    "divyield",
    "gross_profit",
    "roe",
    "asset_growth",
    "leverage",
    "accruals",
    "mkt_12m",
    "mkt_vol_12m",
    "down_market",
]

continuous_predictors = [x for x in numeric_predictors if x != "down_market"]
binary_predictors = ["down_market"]
categorical_predictors = ["ff49_code"]

feature_columns = (
    continuous_predictors
    + binary_predictors
    + categorical_predictors
)

print("Numeric predictors:", len(numeric_predictors))
print("Raw feature columns:", len(feature_columns))


Numeric predictors: 18
Raw feature columns: 19


### Construct the target

The target is next-month excess return, \(r^e_{i,t+1}\). A target is retained only when the next observation for the same stock is exactly one calendar month later.


In [4]:
# A.4.4 Construct next-month excess return

analysis = panel.sort_values(["permno", "date"]).copy()

analysis["next_date"] = analysis.groupby("permno")["date"].shift(-1)
analysis["ret_excess_t1"] = analysis.groupby("permno")["ret_excess_t"].shift(-1)

analysis["is_consecutive_next_month"] = (
    analysis["next_date"].dt.to_period("M")
    == analysis["date"].dt.to_period("M") + 1
)

analysis.loc[
    ~analysis["is_consecutive_next_month"],
    "ret_excess_t1"
] = np.nan

analysis_valid = analysis.loc[
    analysis["ret_excess_t1"].notna()
].copy()

print("Rows with valid next-month targets:", len(analysis_valid))


Rows with valid next-month targets: 197016


### Required chronological split

Only the training and validation samples are used in A.4:

- Training: January 1990–December 2014
- Validation: January 2015–December 2018

The test sample is identified only for an observation-count check and is **not used for tuning or performance measurement here**.


In [5]:
# A.4.5 Create the fixed samples

train = analysis_valid.loc[
    (analysis_valid["date"] >= "1990-01-01")
    & (analysis_valid["date"] <= "2014-12-31")
].copy()

validation = analysis_valid.loc[
    (analysis_valid["date"] >= "2015-01-01")
    & (analysis_valid["date"] <= "2018-12-31")
].copy()

test = analysis_valid.loc[
    (analysis_valid["date"] >= "2019-01-01")
    & (analysis_valid["date"] <= "2022-11-30")
].copy()

split_summary = pd.DataFrame({
    "Sample": ["Training", "Validation", "Test (untouched)"],
    "Months": [
        train["date"].dt.to_period("M").nunique(),
        validation["date"].dt.to_period("M").nunique(),
        test["date"].dt.to_period("M").nunique(),
    ],
    "Stock-month rows": [
        len(train),
        len(validation),
        len(test),
    ],
})

split_summary


,Sample,Months,Stock-month rows
0,Training,300,149334
1,Validation,48,24051
2,Test (untouched),47,23631


### Preprocessing

Preprocessing is fitted using the **training sample only** and then applied unchanged to validation data.

- Continuous predictors: median imputation + standardisation.
- `down_market`: most-frequent imputation, retained as 0/1.
- `ff49_code`: one-hot encoding with one reference category omitted.

No validation information is used to estimate preprocessing parameters.


In [6]:
# A.4.6 Fit preprocessing on training only

continuous_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

categorical_pipeline = Pipeline(steps=[
    ("onehot", OneHotEncoder(
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("continuous", continuous_pipeline, continuous_predictors),
        ("binary", binary_pipeline, binary_predictors),
        ("industry", categorical_pipeline, categorical_predictors),
    ],
    remainder="drop",
)

X_train_raw = train[feature_columns]
X_validation_raw = validation[feature_columns]

y_train = train["ret_excess_t1"].to_numpy()
y_validation = validation["ret_excess_t1"].to_numpy()

preprocessor.fit(X_train_raw)

X_train = preprocessor.transform(X_train_raw)
X_validation = preprocessor.transform(X_validation_raw)

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_validation.shape)


Training matrix: (149334, 64)
Validation matrix: (24051, 64)


## Hyperparameter search rule

A 27-combination grid is used:

- `n_estimators`: 5, 10, 25
- `learning_rate`: 0.05, 0.10, 0.15
- `max_depth`: 1, 2, 3

The loss function is fixed at squared error.

`max_features="sqrt"` is fixed **before** validation as a computational and regularisation setting. It reduces the number of features considered at each tree split while leaving the full predictor set available across the ensemble. It is not selected using validation performance.

The grid is deliberately modest enough to run reliably while still testing different ensemble sizes, shrinkage rates, and interaction depths.


In [7]:
# A.4.7 Define the reproducible tuning grid

param_grid = {
    "n_estimators": [5, 10, 25],
    "learning_rate": [0.05, 0.10, 0.15],
    "max_depth": [1, 2, 3],
}

grid_combinations = list(itertools.product(
    param_grid["n_estimators"],
    param_grid["learning_rate"],
    param_grid["max_depth"],
))

print("Number of candidate specifications:", len(grid_combinations))
print("Grid:", param_grid)


Number of candidate specifications: 27
Grid: {'n_estimators': [5, 10, 25], 'learning_rate': [0.05, 0.1, 0.15], 'max_depth': [1, 2, 3]}


### Validation criterion

For every candidate:

1. fit Gradient Boosting on the training sample;
2. predict the validation sample;
3. calculate pooled validation RMSE;
4. retain the candidate with the lowest validation RMSE.

The winning model object is kept directly from the tuning loop, so there is **no post-selection re-fit**.


In [8]:
# A.4.8 Tune Gradient Boosting using VALIDATION RMSE ONLY

tuning_rows = []

best_rmse = np.inf
best_params = None
selected_gb_model = None
selected_validation_pred = None

for n_estimators, learning_rate, max_depth in grid_combinations:

    model = GradientBoostingRegressor(
        loss="squared_error",
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        max_features="sqrt",
        random_state=RANDOM_STATE,
    )

    # Fit candidate on TRAINING ONLY
    model.fit(X_train, y_train)

    # Evaluate candidate on VALIDATION ONLY
    validation_pred = model.predict(X_validation)

    validation_mse = mean_squared_error(
        y_validation,
        validation_pred
    )
    validation_rmse = np.sqrt(validation_mse)

    tuning_rows.append({
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "validation_mse": validation_mse,
        "validation_rmse": validation_rmse,
    })

    # Retain the already-fitted winning training model
    if validation_rmse < best_rmse:
        best_rmse = validation_rmse
        best_params = {
            "n_estimators": n_estimators,
            "learning_rate": learning_rate,
            "max_depth": max_depth,
        }
        selected_gb_model = model
        selected_validation_pred = validation_pred.copy()

tuning_results = pd.DataFrame(tuning_rows).sort_values(
    "validation_rmse",
    ascending=True
).reset_index(drop=True)

tuning_results


,n_estimators,learning_rate,max_depth,validation_mse,validation_rmse
0,10,0.15,1,0.005901,0.076816
1,10,0.10,1,0.005901,0.076817
2,5,0.15,1,0.005901,0.076821
3,10,0.05,1,0.005901,0.076821
4,5,0.10,1,0.005902,0.076823
5,5,0.05,1,0.005902,0.076826
6,25,0.05,1,0.005903,0.076831
7,10,0.05,3,0.005905,0.076847
8,5,0.05,3,0.005906,0.076850
9,25,0.10,1,0.005907,0.076856


### Winning specification

The next cell reports the hyperparameters associated with the minimum pooled validation RMSE. Selection is based only on the validation sample; no test-period outcome is consulted.


In [9]:
# A.4.9 Report the winning Gradient Boosting specification

best_result = tuning_results.iloc[0]

selected_summary = pd.DataFrame({
    "Item": [
        "Selected n_estimators",
        "Selected learning_rate",
        "Selected max_depth",
        "Validation MSE",
        "Validation RMSE",
        "Training observations",
        "Validation observations",
        "Model refitted after selection?",
    ],
    "Value": [
        int(best_result["n_estimators"]),
        float(best_result["learning_rate"]),
        int(best_result["max_depth"]),
        float(best_result["validation_mse"]),
        float(best_result["validation_rmse"]),
        len(train),
        len(validation),
        "No",
    ],
})

selected_summary


,Item,Value
0,Selected n_estimators,10
1,Selected learning_rate,0.15
2,Selected max_depth,1
3,Validation MSE,0.005901
4,Validation RMSE,0.076816
5,Training observations,149334
6,Validation observations,24051
7,Model refitted after selection?,No


In [10]:
# A.4.10 Confirm the retained model is the winning training-fitted model

print("Winning hyperparameters:", best_params)
print("Lowest validation RMSE:", round(best_rmse, 9))
print("Selected model n_estimators:", selected_gb_model.n_estimators)
print("Selected model learning_rate:", selected_gb_model.learning_rate)
print("Selected model max_depth:", selected_gb_model.max_depth)
print("Selected model was NOT re-fitted on training + validation.")


Winning hyperparameters: {'n_estimators': 10, 'learning_rate': 0.15, 'max_depth': 1}
Lowest validation RMSE: 0.076816164
Selected model n_estimators: 10
Selected model learning_rate: 0.15
Selected model max_depth: 1
Selected model was NOT re-fitted on training + validation.


### Retain validation predictions for later evaluation

The selected model's validation predictions are stored below. A.5 will use these predictions to calculate the required richer-model validation RMSE and mean monthly Spearman rank correlation.

No test predictions are generated in this tuning notebook.


In [11]:
# A.4.11 Store selected-model validation predictions

selected_validation_predictions = validation[
    ["date", "permno", "ticker", "ret_excess_t1"]
].copy()

selected_validation_predictions = selected_validation_predictions.rename(
    columns={"ret_excess_t1": "actual_excess_return_t1"}
)

selected_validation_predictions[
    "gb_pred_excess_return_t1"
] = selected_validation_pred

print("Selected-model validation predictions:",
      len(selected_validation_predictions))
print("Missing predictions:",
      int(selected_validation_predictions[
          "gb_pred_excess_return_t1"
      ].isna().sum()))

selected_validation_predictions.head()


Selected-model validation predictions: 24051
Missing predictions: 0


,date,permno,ticker,actual_excess_return_t1,gb_pred_excess_return_t1
539,2015-01-30,10104,ORCL,0.046073,0.007218
540,2015-02-27,10104,ORCL,-0.015290,0.007218
541,2015-03-31,10104,ORCL,0.014368,0.007218
542,2015-04-30,10104,ORCL,-0.002980,0.007218
543,2015-05-29,10104,ORCL,-0.073350,0.007218


## A.4 Summary

The reproducible 27-specification Gradient Boosting grid was estimated using the training sample only and ranked using pooled validation RMSE.

The selected specification is the candidate with the lowest validation RMSE. The corresponding **already-fitted training model is retained directly from the tuning loop** and is not re-estimated on the combined training and validation samples.

The test period remains untouched. The retained model and validation predictions are ready for A.5 prediction evaluation.
